# Paper 07 · Generative Adversarial Networks

**Citation:** Ian Goodfellow et al., “Generative Adversarial Nets” (2014).

**Paper:** https://arxiv.org/abs/1406.2661

> **Scale gap:** We learn a 2-D mixture of Gaussians rather than images, making mode coverage visible.

## Mathematical Framework

Before reproducing the paper experimentally, work through the relevant mathematical companions:

- [Math 03 · Probability & Bayes](../../math/03_probability_bayes.ipynb)
- [Math 05 · Information Theory](../../math/05_information_theory.ipynb)
- [Math 06 · Optimization](../../math/06_optimization.ipynb)
- [Math 12 · Generative-Model Mathematics](../../math/12_generative_models_math.ipynb)

For the paper defense, be able to explain the **objective, derivation, assumptions, and why the reported mechanism should follow mathematically**, not just what the code did.

## Before you read
1. What does the discriminator estimate?
2. Why can the generator receive a useful gradient without an explicit likelihood?
3. What is mode collapse?

## Central claim
A generator can learn a data distribution by competing against a discriminator in a minimax game.

## Target distribution

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from torch import nn
torch.manual_seed(0); np.random.seed(0)
centers=torch.tensor([[2.,0.],[-2.,0.],[0.,2.],[0.,-2.]])
def real_batch(n):
    ids=torch.randint(0,4,(n,))
    return centers[ids]+.2*torch.randn(n,2)
r=real_batch(1500)
plt.scatter(r[:,0],r[:,1],s=4,alpha=.3); plt.title("Real distribution"); plt.axis("equal"); plt.show()

## GAN training
The generator/discriminator are intentionally tiny.

In [ ]:
G=nn.Sequential(nn.Linear(2,64),nn.ReLU(),nn.Linear(64,64),nn.ReLU(),nn.Linear(64,2))
D=nn.Sequential(nn.Linear(2,64),nn.ReLU(),nn.Linear(64,1))
og=torch.optim.Adam(G.parameters(),lr=.001); od=torch.optim.Adam(D.parameters(),lr=.001)
bce=nn.BCEWithLogitsLoss(); checkpoints={}
for step in range(1200):
    real=real_batch(128); fake=G(torch.randn(128,2)).detach()
    od.zero_grad(); ld=bce(D(real),torch.ones(128,1))+bce(D(fake),torch.zeros(128,1)); ld.backward(); od.step()
    og.zero_grad(); gen=G(torch.randn(128,2)); lg=bce(D(gen),torch.ones(128,1)); lg.backward(); og.step()
    if step in [0,100,400,1199]:
        with torch.no_grad(): checkpoints[step]=G(torch.randn(1000,2)).numpy()
print("final losses D/G",float(ld),float(lg))

## Figure-inspired learning trajectory

In [ ]:
fig,axs=plt.subplots(1,4,figsize=(12,3))
for ax,(step,s) in zip(axs,checkpoints.items()):
    ax.scatter(s[:,0],s[:,1],s=3); ax.set_title(f"step {step}"); ax.set_xlim(-3,3); ax.set_ylim(-3,3)
plt.tight_layout(); plt.show()

### Ablation
Change the discriminator learning rate to 10× the generator learning rate, then reverse it. Compare mode coverage and stability.

## Ablation table
Fill this after running the experiments.

| Variant | Metric / observation | What changed? | Why? |
|---|---:|---|---|
| Baseline |  |  |  |
| Ablation 1 |  |  |  |
| Ablation 2 |  |  |  |

## Defend the paper
Answer without looking back at the notebook:
1. What problem existed before this work?
2. What was actually new?
3. What evidence in your reproduction supports the central claim?
4. What does your reduced-scale reproduction **not** establish?
5. Which idea from this paper survived into modern systems?
6. What experiment would you run next?